In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/fashion_ecommerce/dataset_fashion_store_sales.csv")
df.head()

,sale_id,channel,discounted,total_amount,sale_date,customer_id,country
0,10,E-commerce,0,299.70,2025-05-21,195,France
1,100,App Mobile,0,681.05,2025-04-21,518,Germany
2,1000,E-commerce,0,324.50,2025-05-20,439,Germany
3,1001,E-commerce,0,287.85,2025-04-05,349,Germany
4,1003,App Mobile,0,430.64,2025-06-06,727,Portugal


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 905 entries, 0 to 904
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sale_id       905 non-null    int64  
 1   channel       905 non-null    object 
 2   discounted    905 non-null    int64  
 3   total_amount  905 non-null    float64
 4   sale_date     905 non-null    object 
 5   customer_id   905 non-null    int64  
 6   country       905 non-null    object 
dtypes: float64(1), int64(3), object(3)
memory usage: 49.6+ KB


# Definir el modelo de datos estandarizado

In [4]:
modelo_datos = {
    "customer_id": {"required": True,  "dtype": "string"},
    "transaction_id": {"required": True,  "dtype": "string"},
    "transaction_date": {"required": True,  "dtype": "datetime64[ns]"},
    "transaction_amount": {"required": True,  "dtype": "float64"},
}

# Aqui definimos los nombres de las columnas del dataset, para estandarizarlos al modelo de datos común.
mapeo_cliente = {
    "customer_id": "customer_id",
    "transaction_id": "transaction_id",
    "transaction_date": "transaction_date",
    "transaction_amount": "transaction_amount",
}


In [5]:
import pandas as pd

def standardize_transactions(
    df: pd.DataFrame,
    mapping: dict,
    schema: dict,
    *,
    date_format: str | None = None,
    drop_invalid: bool = False
) -> pd.DataFrame:
    """
    Convierte un dataset de transacciones del cliente a tu modelo canónico.

    Params
    ------
    df : DataFrame original del cliente
    mapping : dict {canonical_col: source_col}
    schema : dict con required/dtype por columna canónica
    date_format : opcional, formato de fecha si viene como texto (ej: "%Y-%m-%d")
    drop_invalid : si True, elimina filas con problemas críticos; si False, lanza error

    Returns
    -------
    DataFrame con columnas canónicas y tipos estandarizados
    """

    # 1) Validar que existen las columnas fuente necesarias
    missing_sources = [src for src in mapping.values() if src not in df.columns]
    if missing_sources:
        raise ValueError(f"Faltan columnas en el dataset fuente: {missing_sources}")

    # 2) Seleccionar + renombrar a canónico
    out = df[list(mapping.values())].copy()
    out.columns = list(mapping.keys())

    # 3) Validar required canónicas (por si alguien pasa mapping incompleto)
    required = [k for k, v in schema.items() if v.get("required")]
    missing_required = [c for c in required if c not in out.columns]
    if missing_required:
        raise ValueError(f"Faltan columnas canónicas requeridas: {missing_required}")

    # 4) Cast tipos
    # IDs
    out["customer_id"] = out["customer_id"].astype("string").str.strip()
    out["transaction_id"] = out["transaction_id"].astype("string").str.strip()

    # Fecha
    if date_format:
        out["transaction_date"] = pd.to_datetime(out["transaction_date"], format=date_format, errors="coerce")
    else:
        out["transaction_date"] = pd.to_datetime(out["transaction_date"], errors="coerce")

    # Importe
    out["transaction_amount"] = pd.to_numeric(out["transaction_amount"], errors="coerce")

    # 5) Checks de calidad mínimos
    errors = []
    if out["customer_id"].isna().any():
        errors.append("customer_id tiene nulos tras estandarizar")
    if out["transaction_id"].isna().any():
        errors.append("transaction_id tiene nulos tras estandarizar")
    if out["transaction_date"].isna().any():
        errors.append("transaction_date tiene valores no parseables (NaT)")
    if out["transaction_amount"].isna().any():
        errors.append("transaction_amount tiene valores no numéricos (NaN)")

    # Duplicados críticos (muy importante para F)
    dup = out.duplicated(subset=["transaction_id"]).sum()
    if dup > 0:
        errors.append(f"transaction_id tiene {dup} duplicados (revisar granularidad / líneas de pedido)")

    if errors:
        if drop_invalid:
            # Elimina filas inválidas (mínimo viable)
            out = out.dropna(subset=["customer_id", "transaction_id", "transaction_date", "transaction_amount"])
            out = out.drop_duplicates(subset=["transaction_id"])
        else:
            raise ValueError(" | ".join(errors))

    return out


In [7]:
df_raw = pd.read_csv("ventas_cliente_x.csv")

df_canon = standardize_transactions(
    df_raw,
    mapping=modelo_datos,
    schema=mapeo_cliente,
    date_format="%d/%m/%Y"  # si aplica
)

df_canon.head()


FileNotFoundError: [Errno 2] No such file or directory: 'ventas_cliente_x.csv'

In [6]:
data = df.copy()
data["order_purchase_timestamp"] = pd.to_datetime(data["order_purchase_timestamp"], errors="coerce")

KeyError: 'order_purchase_timestamp'

In [15]:
import pandas as pd
import numpy as np

# =========================
# Parámetros de análisis
# =========================
DESDE = None        # Ej: "2017-01-01" o None
HASTA = None        # Ej: "2018-01-01" o None
P = 3               # Nº de últimas transacciones para R'

# =========================
# Carga de datos
# =========================
# df = pd.read_csv("orders.csv")

required_cols = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

# =========================
# Preprocesado fechas
# =========================
df = data.copy()
df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

# Filtramos pedidos válidos (ajusta si lo necesitas)
df = df[df["order_status"] == "delivered"]

# Eliminamos filas sin fecha válida
df = df.dropna(subset=["order_purchase_timestamp"])

# =========================
# Definición automática del intervalo
# =========================
if DESDE is None or DESDE == "":
    desde_dt = df["order_purchase_timestamp"].min()
else:
    desde_dt = pd.to_datetime(DESDE)

if HASTA is None or HASTA == "":
    hasta_dt = df["order_purchase_timestamp"].max()
else:
    hasta_dt = pd.to_datetime(HASTA)

# =========================
# Filtrado por periodo
# =========================
df_period = df[
    (df["order_purchase_timestamp"] >= desde_dt) &
    (df["order_purchase_timestamp"] <= hasta_dt)
]

if df_period.empty:
    out = pd.DataFrame(
        columns=["customer_id", "L_days", "Rprime_days", "F"]
    )
    print(out)
else:
    # =========================
    # F - Frequency
    # =========================
    F = (
        df_period
        .groupby("customer_id")["order_id"]
        .nunique()
        .rename("F")
    )

    # =========================
    # L - Length
    # =========================
    g = df_period.groupby("customer_id")["order_purchase_timestamp"]
    first_purchase = g.min()
    last_purchase = g.max()

    L_days = (
        (last_purchase - first_purchase)
        .dt.total_seconds()
        .div(86400)
        .rename("L_days")
    )

    # =========================
    # R′ - Recency mejorada
    # =========================
    df_sorted = df_period.sort_values(
        ["customer_id", "order_purchase_timestamp"]
    )

    last_p = df_sorted.groupby("customer_id").tail(P)

    last_p = last_p.assign(
        diff_days=(hasta_dt - last_p["order_purchase_timestamp"])
        .dt.total_seconds()
        .div(86400)
    )

    Rprime_days = (
        last_p
        .groupby("customer_id")["diff_days"]
        .mean()
        .rename("Rprime_days")
    )

    # =========================
    # Resultado final
    # =========================
    out = (
        pd.concat([L_days, Rprime_days, F], axis=1)
        .reset_index()
    )

    out["L_days"] = out["L_days"].round(2)
    out["Rprime_days"] = out["Rprime_days"].round(2)

    print(out.head(20))



                         customer_id  L_days  Rprime_days  F
0   00012a2ce6f8dcda20d059ce98491703     0.0       287.95  1
1   000161a058600d5901f007fab4c27140     0.0       409.22  1
2   0001fd6190edaaf884bcaf3d49edf079     0.0       547.16  1
3   0002414f95344307404f0ace7a26f1d5     0.0       378.08  1
4   000379cdec625522490c315e70c7a9fb     0.0       149.05  1
5   0004164d20a9e969af783496f3408652     0.0       504.27  1
6   000419c5494106c306a97b5635748086     0.0       179.88  1
7   00046a560d407e99b969756e0b10f282     0.0       254.16  1
8   00050bf6e01e69d5c0fd612f1bcfb69c     0.0       345.96  1
9   000598caf2ef4117407665ac33275130     0.0        18.12  1
10  0005aefbb696d34b3424dccd0a0e9fd0     0.0        70.22  1
11  00062b33cb9f6fe976afdcff967ea74d     0.0       531.64  1
12  00066ccbe787a588c52bd5ff404590e3     0.0       203.95  1
13  00072d033fe2e59061ae5c3aff1a2be5     0.0       362.23  1
14  0009a69b72033b2d0ec8c69fc70ef768     0.0       488.06  1
15  000bf8121c3412d3057d